# ChromaNeRV CVPR Phase 2
Run preflight first. Training and latency are opt-in. Use an A100 runtime for the latency benchmark.


In [ ]:
RUN_TRAINING = False
RUN_LATENCY = False
ONLY_SEQUENCE = 'HoneyBee'
ONLY_VARIANT = 'rgbsplit_yc_a320'
MAX_RUNS = 1
EPOCHS = 150
BUNNY_ZIP = ''  # Original Bunny 132-frame ZIP in MyDrive
BUNNY_COMMAND = ''  # Original Bunny command.txt path
UVG_EXPECTED_SHA256 = ''  # Optional trusted hash from Phase 1 audit
CHECKPOINT_EXPECTED_SHA256 = ''  # Optional trusted archive hash
LATENCY_MANIFEST = ''  # Optional existing six-checkpoint manifest
LATENCY_EXTRA_ROOT = ''  # Optional second extracted original checkpoint archive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
drive_root = Path('/content/drive/MyDrive')
persistent = drive_root / 'ChromaNeRV_CVPR_Phase2'
persistent.mkdir(parents=True, exist_ok=True)
local = Path('/content/chromanerv_phase2')
local.mkdir(exist_ok=True)
repo = Path('/content/ChromaHNeRV')
if not repo.exists(): subprocess.run(['git','clone','https://github.com/HannanAkhtar/ChromaHNeRV.git',str(repo)], check=True)
branch = 'experiment/cvpr27-phase2-controls-latency'
subprocess.run(['git','-C',str(repo),'fetch','origin',branch], check=True)
subprocess.run(['git','-C',str(repo),'checkout',branch], check=True)
subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',branch], check=True)
sha = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip()
print('Repository commit:', sha)
subprocess.run([sys.executable,str(repo/'phase2_colab_setup.py'),'--output',str(persistent/'phase2_environment.json')], check=True)
environment = subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True)
(persistent/'phase2_environment.txt').write_text(f'commit={sha}\n'+environment)


In [ ]:
sys.path.insert(0,str(repo))
from phase2_archive import extract_verified
def locate_zip(name):
    found = list(drive_root.rglob(name))
    if len(found) != 1: raise FileNotFoundError(f'Expected one {name} in MyDrive, found {len(found)}')
    return found[0]
uvg_root, uvg_audit = extract_verified(locate_zip('ChromaNeRV_UVG132.zip'), local/'uvg_archive', UVG_EXPECTED_SHA256 or None)
checkpoint_root, checkpoint_audit = extract_verified(locate_zip('ChromaNeRV_CVPR_Checkpoints_20261115_021417.zip'), local/'checkpoints_archive', CHECKPOINT_EXPECTED_SHA256 or None)
(persistent/'phase2_archive_audit.json').write_text(json.dumps({'uvg':uvg_audit,'checkpoints':checkpoint_audit},indent=2))
print('Archive hashes are provenance only when compared with trusted expected hashes.')
print('UVG:', uvg_root, 'Checkpoints:', checkpoint_root)


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',str(repo/'tests')], cwd=repo, check=True)
from run_cvpr27_phase2 import SEQUENCES, preflight
from uvg_utils import discover_image_files
from PIL import Image
def find_uvg_root(root):
    candidates = [p for p in root.rglob('HoneyBee') if p.is_dir() and len(discover_image_files(p)) == 132]
    if len(candidates) != 1: raise ValueError(f'Expected one 132-frame HoneyBee directory, found {candidates}')
    return candidates[0].parent
uvg_frames = find_uvg_root(uvg_root)
manifest = json.loads((repo/'configs/cvpr27_phase2_132.json').read_text())
for sequence in SEQUENCES[:-1]: preflight({'sequence':sequence}, uvg_frames/sequence, manifest[sequence])
bunny_frames = ''
if BUNNY_ZIP:
    bunny_root, bunny_audit = extract_verified(Path(BUNNY_ZIP),local/'bunny_archive')
    candidates = [p for p in [bunny_root,*bunny_root.rglob('*')] if p.is_dir() and sum(f.is_file() and f.suffix.lower() in {'.png','.jpg','.jpeg'} for f in p.iterdir()) == 132]
    if len(candidates) != 1: raise ValueError(f'Expected one original 132-frame Bunny directory, found {candidates}')
    bunny_frames = str(candidates[0])
    preflight({'sequence':'Bunny'},Path(bunny_frames),manifest['Bunny'])
base = [sys.executable,str(repo/'run_cvpr27_phase2.py'),'--uvg-root',str(uvg_frames),'--output-root',str(local/'runs'),'--backup-root',str(persistent/'runs')]
subprocess.run(base + ['--dry-run'], cwd=repo, check=True)


In [ ]:
if RUN_TRAINING:
    command = base + ['--max-runs',str(MAX_RUNS),'--epochs',str(EPOCHS),'--strict-backup']
    if ONLY_SEQUENCE: command += ['--only-sequence',ONLY_SEQUENCE]
    if ONLY_VARIANT: command += ['--only-variant',ONLY_VARIANT]
    if bunny_frames and BUNNY_COMMAND: command += ['--bunny-root',bunny_frames,'--bunny-command',BUNNY_COMMAND]
    subprocess.run(command, cwd=repo, check=True)
else: print('Training disabled. Set RUN_TRAINING=True after preflight.')


In [ ]:
if RUN_LATENCY:
    command = [sys.executable,str(repo/'benchmark_decoder_latency.py'),'--frames',str(uvg_frames/'HoneyBee'),'--output',str(persistent/'latency')]
    if LATENCY_MANIFEST: command += ['--manifest',LATENCY_MANIFEST]
    else:
        command += ['--archive-root',str(checkpoint_root)]
        if LATENCY_EXTRA_ROOT: command += ['--archive-root',LATENCY_EXTRA_ROOT]
    subprocess.run(command,cwd=repo,check=True)
else: print('Latency disabled. Set RUN_LATENCY=True on an idle A100.')
